# The Documents ChatBot Architecture

## 前置条件
- ChatGPT 账号：https://foofish.net/how-to-registe-chatGPT.html
- 支付平台：https://nobepay.com/app/login
- 付款方式: https://51.ruyo.net/18298.html

## 架构设计
[The Documents ChatBot Architecture](https://excalidraw.com/#json=SvWicYQQNysuRRG1TgkmO,gCEvhkWNw0CTkGj82orG6w)
![](./static/The_Docs_ChatBot_architecture.svg)

## LangChain 介绍
- 文档链接：https://python.langchain.com/en/latest/getting_started/getting_started.html
- 项目链接：https://github.com/hwchase17/langchain


### 支持的 [Vectorstores](https://python.langchain.com/en/latest/modules/indexes/vectorstores.html)

#### Pinecone 介绍
Pinecone makes it easy to build high-performance vector search applications. It’s a managed, cloud-native vector database with a simple API and no infrastructure hassles.
Pinecone has the following attributes:
- Fast: Get ultra-low query latency at any scale, even with billions of items.
- Fresh: Get live index updates when you add, edit, or delete data.
- Filtered: Combine vector search with metadata filters for more relevant, faster results.
- Fully managed: Get started, use, and scale with ease, while we keep things running smoothly and securely.

#### Qdrant 介绍 
Qdrant (read: quadrant ) is a vector similarity search engine. It provides a production-ready service with a convenient API to store, search, and manage points - vectors with an additional payload. Qdrant is tailored to extended filtering support. It makes it useful for all sorts of neural network or semantic-based matching, faceted search, and other applications.

## 代码示例

本次示例使用 `Pinecone` 作为矢量数据库进行数据存储。

Pinecone SaaS环境地址： https://app.pinecone.io/ ，执行代码前需要完成SaaS环境账号注册。


### Step1. 将文档  embeddings 并将结果写入 pinecone


1. 准备相关变量

我们使用`configparser`管理变量，变量保存路径为`./lcd.conf` 文件。

拷贝目录下 lcd.conf.sample 文件到 lcd.conf。

```bash
cp lcd.conf.sample lcd.conf
```

根据实际情况填写变量信息

- Openai token 获取：https://platform.openai.com/account/api-keys
- Pinecone 访问信息： 在`API Keys`选项卡下获取相关信息。


In [ ]:
from configparser import ConfigParser

conf = ConfigParser()
conf.read("./lcd.conf")
OPENAI_API_KEY = conf["conf"]["OPENAI_API_KEY"]

pinecone_api_key = conf.get("conf", "pinecone_api_key")
pinecone_environment = conf.get("conf","pinecone_environment")

In [2]:
# 2. 加载txt格式文件，并将文件进行分割

# 理论上文件可以是 PDF / markdown 或任何其他文档格式，但是需要自己实现 parser 进行文档的分析。

from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Pinecone
from langchain.document_loaders import TextLoader
loader = TextLoader('./static/The_20th_National_Congress.txt')
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs = text_splitter.split_documents(documents)

In [ ]:
# 3. 将 docs 进行 embeddings 并将结果写入 pinecone

import pinecone 

# initialize pinecone
pinecone.init(
    api_key=pinecone_api_key,  # find at app.pinecone.io
    environment=pinecone_environment,  # next to api key in console
)

index_name = "lcd-demo"
namespace_name = "lcd-demo"

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
vectorstores = Pinecone.from_documents(docs, embeddings, index_name=index_name, namespace=namespace_name)

In [ ]:
# 4. 测试文档搜索

namespace_name = "lcd-demo"
index = pinecone.Index("lcd-demo")
Pinecone_search = Pinecone(index, embeddings.embed_query, "text", namespace_name)


query = "会议如何看待科教兴国战略"
docs = Pinecone_search.similarity_search(query)
docs[0].page_content

### Step2.  将查询结果与问题进行结合（Combine）

In [ ]:
# 1. 准备模板
# 模板会将查询结果和问题进行合并，并且介入一些提示词。

from langchain.prompts.chat import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain.chains.conversational_retrieval.prompts import CONDENSE_QUESTION_PROMPT

chat_combine_template = """
You are a DocsGPT, friendly and helpful AI assistant by Arc53 that provides help with documents. 
Use the following pieces of context to help answer the users question. 
If its not relevant to the question, provide friendly responses.
----------------
{summaries}
"""

messages_combine = [
        SystemMessagePromptTemplate.from_template(chat_combine_template),
        HumanMessagePromptTemplate.from_template("{question}")
]

p_chat_combine = ChatPromptTemplate.from_messages(messages_combine)

In [ ]:
# 2. 使用ChatOpenAI作为大语言模型

from langchain.chat_models import ChatOpenAI
llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY)

In [ ]:
# 3. 创建Chain
# 创建chain 将大语言模型、问题模板、向量数据库进行结合。

from langchain.chains.question_answering import load_qa_chain
from langchain.chains import LLMChain, ConversationalRetrievalChain
doc_chain = load_qa_chain(llm, chain_type="map_reduce", combine_prompt=p_chat_combine)

question_generator = LLMChain(llm=llm, prompt=CONDENSE_QUESTION_PROMPT)

chain = ConversationalRetrievalChain(
                retriever=Pinecone_search.as_retriever(k=5),
                question_generator=question_generator,
                combine_docs_chain=doc_chain,
                return_source_documents=True
            )

In [ ]:
# 4. 查询解问题并输出结果

from pprint import pprint

question = "中国共产党第二十次全国代表大会，在军队建设，军队发展、祖国统一方面提出了哪些举措？"
chat_history = []
result = chain({"question": question, "chat_history": chat_history})
pprint(result)

5. 直接在ChatGPT 中询问无法查到答案

![](./static/q1.png)

## DocsGPT

项目地址： https://github.com/arc53/DocsGPT

DocsGPT is a cutting-edge open-source solution that streamlines the process of finding information in project documentation. With its integration of the powerful GPT models, developers can easily ask questions about a project and receive accurate answers.

Say goodbye to time-consuming manual searches, and let DocsGPT help you quickly find the information you need. Try it out and see how it revolutionizes your project documentation experience. Contribute to its development and be a part of the future of AI-powered assistance.

## Auto-GPT

项目地址： https://github.com/Significant-Gravitas/Auto-GPT

Google Sdearch: [Google Sdearch Token 申请](https://python.langchain.com/en/latest/modules/agents/tools/examples/google_search.html?highlight=google)

![](./static/autoGPT.png)